# RSA algorithm  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 5.4.b: RSA algorithm

**Skill:** Mitigate Risk · **Maps to:** 5.4.A, 5.4.B · **Language:** Python 3 (pytest)

## Problem
A small internal tool needs to demonstrate RSA end to end on textbook-scale numbers: derive a
key pair from two primes and a chosen public exponent, encrypt a small integer message with the
public key, and decrypt it back with the private key. Implement the key derivation and the
modular-exponentiation encrypt/decrypt operations directly, you may use Python's built-in
`pow(base, exp, mod)` for the modular exponentiation itself, but the extended Euclidean
algorithm used to derive the private exponent `d` must be implemented by hand.

## Function contract
Implement, in the code editor:
- `egcd(a: int, b: int) -> tuple[int, int, int]`, extended Euclidean algorithm; returns
  `(g, x, y)` such that `a*x + b*y == g == gcd(a, b)`.
- `mod_inverse(e: int, phi_n: int) -> int`, the modular inverse `d` of `e` mod `phi_n`, i.e.
  `(e * d) % phi_n == 1`. Raises `ValueError` if no inverse exists (`gcd(e, phi_n) != 1`).
- `rsa_keygen(p: int, q: int, e: int) -> dict`, derives `n`, `phi_n`, and `d` from primes `p`,
  `q` and public exponent `e`. Returns `{"n": n, "phi_n": phi_n, "e": e, "d": d}`. Raises
  `ValueError` if `e` does not satisfy `1 < e < phi_n`.
- `encrypt(m: int, e: int, n: int) -> int`, RSA encrypt: `c = m^e mod n`. Raises `ValueError`
  if `m` is not in `0 <= m < n`.
- `decrypt(c: int, d: int, n: int) -> int`, RSA decrypt: `m = c^d mod n`.
- `solve(data: dict) -> dict`, the blackbox entry point that dispatches to the functions above
  (see the Reference section below).

## Input / Output
- `rsa_keygen(p, q, e)`: `p`, `q` are (small, in this atom's tests) prime integers; `e` is the
  chosen public exponent -> `{"n": int, "phi_n": int, "e": int, "d": int}`.
- `encrypt(m, e, n)`: `m: int`, `e: int`, `n: int` -> ciphertext `int`.
- `decrypt(c, d, n)`: `c: int`, `d: int`, `n: int` -> plaintext `int`.
- `solve(data)`:
  - `{"op": "keygen", "p": int, "q": int, "e": int}` -> `{"n": int, "phi_n": int, "e": int, "d": int}`
  - `{"op": "encrypt", "m": int, "e": int, "n": int}` -> `{"c": int}`
  - `{"op": "decrypt", "c": int, "d": int, "n": int}` -> `{"m": int}`
  - `{"op": "roundtrip", "p": int, "q": int, "e": int, "m": int}` ->
    `{"n": int, "phi_n": int, "d": int, "c": int, "m_decrypted": int}`

## Examples
```
rsa_keygen(3, 11, 7)          -> {"n": 33, "phi_n": 20, "e": 7, "d": 3}
encrypt(5, 7, 33)             -> 14
decrypt(14, 3, 33)            -> 5

solve({"op": "roundtrip", "p": 3, "q": 11, "e": 7, "m": 5})
  -> {"n": 33, "phi_n": 20, "d": 3, "c": 14, "m_decrypted": 5}
```

## Constraints
- Deterministic, offline, standard library only (`pow()` for modular exponentiation is allowed
  and expected; do not import any external crypto library, the extended Euclidean algorithm
  and key derivation must be your own code).
- Must pass all cases in the tests (visible + hidden), including edge cases such as `m = 0`,
  `m` at or beyond `n` (rejected), and `e` values that are not coprime with `phi_n` (rejected).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry points
```python
def egcd(a: int, b: int) -> tuple[int, int, int]
def mod_inverse(e: int, phi_n: int) -> int
def rsa_keygen(p: int, q: int, e: int) -> dict
def encrypt(m: int, e: int, n: int) -> int
def decrypt(c: int, d: int, n: int) -> int
def solve(data: dict) -> dict
```

`solve()` is the blackbox entry point; the other functions are the underlying primitives it is
built on and are tested directly as well.

### Input
- `egcd(a, b)`: non-negative integers.
- `mod_inverse(e, phi_n)`: `e` and `phi_n` are positive integers.
- `rsa_keygen(p, q, e)`: `p`, `q` are prime integers; `e` is the candidate public exponent.
- `encrypt(m, e, n)`: `m`, `e`, `n` are non-negative integers; `m` must satisfy `0 <= m < n`.
- `decrypt(c, d, n)`: `c`, `d`, `n` are non-negative integers.
- `solve(data)`: `data` is a `dict` with `"op"` equal to one of `"keygen"`, `"encrypt"`,
  `"decrypt"`, `"roundtrip"`.
  - `"keygen"`: `{"op": "keygen", "p": int, "q": int, "e": int}`
  - `"encrypt"`: `{"op": "encrypt", "m": int, "e": int, "n": int}`
  - `"decrypt"`: `{"op": "decrypt", "c": int, "d": int, "n": int}`
  - `"roundtrip"`: `{"op": "roundtrip", "p": int, "q": int, "e": int, "m": int}`

### Output (`result`)
- `egcd(a, b)` returns `(g, x, y)` with `a*x + b*y == g == gcd(a, b)`.
- `mod_inverse(e, phi_n)` returns the integer `d` with `(e * d) % phi_n == 1`.
- `rsa_keygen(p, q, e)` returns `{"n": int, "phi_n": int, "e": int, "d": int}`.
- `encrypt(m, e, n)` returns the ciphertext `int` (`m^e mod n`).
- `decrypt(c, d, n)` returns the plaintext `int` (`c^d mod n`).
- `solve(...)` returns:
  - `"keygen"` -> `{"n": int, "phi_n": int, "e": int, "d": int}`
  - `"encrypt"` -> `{"c": int}`
  - `"decrypt"` -> `{"m": int}`
  - `"roundtrip"` -> `{"n": int, "phi_n": int, "d": int, "c": int, "m_decrypted": int}`

### Guarantees / rules
- Pure functions, deterministic, no I/O, no network. Standard library only (`pow()` may be used
  for modular exponentiation; the extended Euclidean algorithm must be implemented directly, not
  imported from a crypto library).
- `mod_inverse` and `rsa_keygen` raise `ValueError` when no valid inverse/exponent exists
  (`gcd(e, phi_n) != 1`, or `e` outside `1 < e < phi_n`).
- `encrypt` raises `ValueError` when `m` is not in `0 <= m < n`.
- For any valid key pair, `decrypt(encrypt(m, e, n), d, n) == m` for every `m` in `0 <= m < n`.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 5.4.b. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests).

Implements small-number textbook RSA directly: key derivation (n, phi_n, d via the
extended Euclidean algorithm) and encrypt/decrypt via modular exponentiation
(Python's built-in pow(), which is stdlib, not an external crypto library).
"""


def egcd(a: int, b: int):
    """Extended Euclidean algorithm. Returns (g, x, y) such that a*x + b*y = g = gcd(a, b)."""
    if b == 0:
        return (a, 1, 0)
    g, x1, y1 = egcd(b, a % b)
    return (g, y1, x1 - (a // b) * y1)


def mod_inverse(e: int, phi_n: int) -> int:
    """Return d, the modular inverse of e mod phi_n, satisfying (e * d) % phi_n == 1.

    Raises ValueError if e and phi_n are not coprime (no inverse exists)."""
    g, x, _ = egcd(e, phi_n)
    if g != 1:
        raise ValueError(f"e={e} has no modular inverse mod phi_n={phi_n} (gcd={g})")
    return x % phi_n


def rsa_keygen(p: int, q: int, e: int) -> dict:
    """Derive full RSA key material from primes p, q and public exponent e.

    Returns {"n": n, "phi_n": phi_n, "e": e, "d": d}.
    """
    n = p * q
    phi_n = (p - 1) * (q - 1)
    if not (1 < e < phi_n):
        raise ValueError(f"e={e} must satisfy 1 < e < phi_n={phi_n}")
    d = mod_inverse(e, phi_n)
    return {"n": n, "phi_n": phi_n, "e": e, "d": d}


def encrypt(m: int, e: int, n: int) -> int:
    """RSA encrypt: c = m^e mod n. m must satisfy 0 <= m < n."""
    if not (0 <= m < n):
        raise ValueError(f"message m={m} must satisfy 0 <= m < n={n}")
    return pow(m, e, n)


def decrypt(c: int, d: int, n: int) -> int:
    """RSA decrypt: m = c^d mod n."""
    return pow(c, d, n)


def solve(data):
    """Blackbox entry point. Dispatches on data['op']:
      - {"op": "keygen", "p": int, "q": int, "e": int}
            -> {"n": int, "phi_n": int, "e": int, "d": int}
      - {"op": "encrypt", "m": int, "e": int, "n": int}
            -> {"c": int}
      - {"op": "decrypt", "c": int, "d": int, "n": int}
            -> {"m": int}
      - {"op": "roundtrip", "p": int, "q": int, "e": int, "m": int}
            -> {"n": int, "phi_n": int, "d": int, "c": int, "m_decrypted": int}
    """
    op = data["op"]
    if op == "keygen":
        return rsa_keygen(data["p"], data["q"], data["e"])
    elif op == "encrypt":
        return {"c": encrypt(data["m"], data["e"], data["n"])}
    elif op == "decrypt":
        return {"m": decrypt(data["c"], data["d"], data["n"])}
    elif op == "roundtrip":
        keys = rsa_keygen(data["p"], data["q"], data["e"])
        c = encrypt(data["m"], keys["e"], keys["n"])
        m_decrypted = decrypt(c, keys["d"], keys["n"])
        return {
            "n": keys["n"],
            "phi_n": keys["phi_n"],
            "d": keys["d"],
            "c": c,
            "m_decrypted": m_decrypted,
        }
    else:
        raise ValueError(f"unknown op: {op!r}")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

import pytest


# ---- visible cases -------------------------------------------------------
def test_keygen_small_example():
    # p=3, q=11, e=7 -> n=33, phi_n=20, d=3 (matches atom.md worked example)
    result = impl.rsa_keygen(3, 11, 7)
    assert result == {"n": 33, "phi_n": 20, "e": 7, "d": 3}


def test_encrypt_matches_worked_example():
    assert impl.encrypt(5, 7, 33) == 14


def test_decrypt_matches_worked_example():
    assert impl.decrypt(14, 3, 33) == 5


def test_encrypt_then_decrypt_roundtrip():
    keys = impl.rsa_keygen(3, 11, 7)
    for m in range(0, 33):
        c = impl.encrypt(m, keys["e"], keys["n"])
        assert impl.decrypt(c, keys["d"], keys["n"]) == m


def test_solve_keygen_op():
    result = impl.solve({"op": "keygen", "p": 3, "q": 11, "e": 7})
    assert result == {"n": 33, "phi_n": 20, "e": 7, "d": 3}


def test_solve_encrypt_op():
    result = impl.solve({"op": "encrypt", "m": 5, "e": 7, "n": 33})
    assert result == {"c": 14}


def test_solve_decrypt_op():
    result = impl.solve({"op": "decrypt", "c": 14, "d": 3, "n": 33})
    assert result == {"m": 5}


def test_solve_roundtrip_op():
    result = impl.solve({"op": "roundtrip", "p": 3, "q": 11, "e": 7, "m": 5})
    assert result == {"n": 33, "phi_n": 20, "d": 3, "c": 14, "m_decrypted": 5}


# ---- hidden / edge cases -------------------------------------------------
def test_keygen_different_primes():
    # p=61, q=53, e=17 (larger textbook-style example)
    result = impl.rsa_keygen(61, 53, 17)
    n = 61 * 53
    phi_n = 60 * 52
    assert result["n"] == n
    assert result["phi_n"] == phi_n
    assert (17 * result["d"]) % phi_n == 1


def test_full_roundtrip_larger_keys():
    keys = impl.rsa_keygen(61, 53, 17)
    m = 1234
    c = impl.encrypt(m, keys["e"], keys["n"])
    assert impl.decrypt(c, keys["d"], keys["n"]) == m
    assert c != m


def test_mod_inverse_correctness():
    d = impl.mod_inverse(7, 20)
    assert (7 * d) % 20 == 1
    assert d == 3


def test_mod_inverse_raises_when_not_coprime():
    with pytest.raises(ValueError):
        impl.mod_inverse(4, 20)  # gcd(4, 20) = 4, no inverse


def test_encrypt_message_zero():
    keys = impl.rsa_keygen(3, 11, 7)
    c = impl.encrypt(0, keys["e"], keys["n"])
    assert c == 0
    assert impl.decrypt(c, keys["d"], keys["n"]) == 0


def test_encrypt_rejects_message_too_large():
    with pytest.raises(ValueError):
        impl.encrypt(33, 7, 33)  # m must be < n


def test_keygen_rejects_invalid_e():
    with pytest.raises(ValueError):
        impl.rsa_keygen(3, 11, 4)  # gcd(4, 20) = 4, not coprime with phi_n


def test_ciphertext_differs_from_plaintext_for_nontrivial_message():
    keys = impl.rsa_keygen(61, 53, 17)
    m = 42
    c = impl.encrypt(m, keys["e"], keys["n"])
    assert c != m


def test_egcd_basic_identity():
    a, b = 7, 20
    g, x, y = impl.egcd(a, b)
    assert g == 1
    assert a * x + b * y == g


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
